<a href="https://colab.research.google.com/github/ethanpwu/Bami-s-Cinder/blob/main/deloitte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install qiskit

In [7]:
!pip install qiskit qiskit-machine-learning qiskit-algorithms pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.5 MB/s eta 0:00:00


Clone Bami-s-Cinder

In [6]:
!git clone https://github.com/ethanpwu/Bami-s-Cinder.git

Cloning into 'Bami-s-Cinder'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 14 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 6.68 MiB | 10.04 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#datasets
wildfire_1 = pd.read_csv('/content/Bami-s-Cinder/WildfireDataset1.csv', encoding='latin1')
wildfire_2 = pd.read_csv('/content/Bami-s-Cinder/WildfireDataset2.csv', encoding='latin1', low_memory=False)
zipcode1 = pd.read_csv('/content/Bami-s-Cinder/ZipcodeInsuranceDataset1.csv', encoding='latin1' ,low_memory=False)
zipcode2 = pd.read_csv('/content/Bami-s-Cinder/ZipcodeInsuranceDataset2.csv', encoding='latin1', low_memory=False)


In [20]:
#fire_occured
wildfire_2['Fire_Occurred'] = wildfire_2['OBJECTID'].notnull().astype(int)

features = ['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']

# 3. Clean the data by dropping rows that are missing our weather features
df_clean = wildfire_2.dropna(subset=features)

# 4. Sample the data
# Quantum simulators take a LONG time to run on standard computers.
# As a beginner, it is best to start small so you can test if your code works.
# Let's randomly select 100 fires and 100 non-fires to train our initial model.
fires = df_clean[df_clean['Fire_Occurred'] == 1].sample(100, random_state=42)
no_fires = df_clean[df_clean['Fire_Occurred'] == 0].sample(100, random_state=42)

# Combine them into one smaller, balanced dataset
df_sample = pd.concat([fires, no_fires])

# 5. Separate the features (X) from the target (y)
X = df_sample[features].values
y = df_sample['Fire_Occurred'].values

# 6. Scale the features for the Quantum Computer
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 7. Split the data into a "Training" set (to teach the model) and a "Testing" set (to quiz it)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Data is clean and ready!")
print(f"We are training the model on {len(X_train)} rows and testing it on {len(X_test)} rows.")

Data is clean and ready!
We are training the model on 160 rows and testing it on 40 rows.


In [22]:
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import VQC
from qiskit.primitives import StatevectorSampler as Sampler

# 1. Get the number of features (we have 3: max temp, min temp, rain)
num_features = X_train.shape[1]

# 2. Build the Feature Map: Translates classical numbers into quantum states
feature_map = ZZFeatureMap(feature_dimension=num_features, reps=1)

# 3. Build the Ansatz: The "learning" part of the circuit with adjustable dials
ansatz = RealAmplitudes(num_qubits=num_features, reps=2)

# 4. Set the Optimizer: The classical helper that adjusts the dials.
# We set maxiter=40 to make it run relatively fast for this test.
optimizer = COBYLA(maxiter=40)

# 5. Define the Sampler: The engine that executes our quantum circuit
sampler = Sampler()

# 6. Put it all together into the VQC Model
vqc = VQC(
    sampler=sampler,
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

# 7. Train the model using our training data
print("Training the Quantum Model... (This might take a minute or two)")
vqc.fit(X_train, y_train)

# 8. Quiz the model on the test data to see how accurate it is
score = vqc.score(X_test, y_test)
print(f"Quantum Model Accuracy on Test Data: {score * 100:.2f}%")

/tmp/ipykernel_20611/1072974575.py:10: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=num_features, reps=1)
/tmp/ipykernel_20611/1072974575.py:13: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(num_qubits=num_features, reps=2)


Training the Quantum Model... (This might take a minute or two)
Quantum Model Accuracy on Test Data: 52.50%
